# Q-factorisation on Gridworld Maze- Baseline, full retrain not regularisation

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import time
from copy import deepcopy


# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from environments.maze_discrete import MazeGridWorld, MazeGoalWrapper
from utils import (TrajectoryReplayBufferDiscrete, evaluate_policy, set_seed, build_goal_batch, get_base_env, collect_valid_states_fourrooms,
                   estimate_fisher_diag, extract_fixed_probe_sa_embedding, extract_mean_sa_embedding, extract_sa_batch_for_isotropy,
                   compute_embedding_drift, collect_weight_snapshot)
from visualisations import visualise_embeddings, visualise_q_table, print_goal_embedding_similarity, plot_full_embedding_dashboard_html
from loss_functions import repulsion_loss_to_memory, sigreg_loss, orthogonal_loss, ewc_regulariser_loss, weight_regulariser_loss
from networks import snapshot_named_parameters, FactorisedDQN_QNetwork
from trainer import dqn_train


DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
class FactorisedDQNQNetworkEnsemble(nn.Module):
    def __init__(
        self,
        obs_dim,
        num_actions,
        goal_dim,
        hidden_dim=128,
        rep_dim=64,
        num_heads=5,
    ):
        super().__init__()
        self.obs_dim = obs_dim
        self.num_actions = num_actions
        self.goal_dim = goal_dim
        self.rep_dim = rep_dim
        self.num_heads = num_heads

        self.sa_encoder = nn.Sequential(
            nn.Linear(obs_dim + num_actions, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, rep_dim),
        )

        self.goal_encoder = nn.Sequential(
            nn.Linear(goal_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, rep_dim),
        )

        self.q_heads = nn.ModuleList([
            nn.Linear(rep_dim, 1) for _ in range(num_heads)
        ])

    def encode_state_action(self, obs, act_onehot):
        x = torch.cat([obs, act_onehot], dim=-1)
        return self.sa_encoder(x)

    def encode_goal(self, goal):
        return self.goal_encoder(goal)

    def forward(self, obs, act_onehot, goal, return_all_heads=False):
        phi = self.encode_state_action(obs, act_onehot)   # [B,D]
        psi = self.encode_goal(goal)                      # [B,D]
        pair = phi * psi                                  # [B,D]

        q_heads = torch.cat([head(pair) for head in self.q_heads], dim=1)  # [B,K]

        if return_all_heads:
            return q_heads
        return q_heads.mean(dim=1, keepdim=True)

    @torch.no_grad()
    def q_values_all_actions_all_heads(self, obs, goal):
        B = obs.shape[0]
        A = self.num_actions

        act_eye = torch.eye(A, device=obs.device)
        act_all = act_eye.unsqueeze(0).expand(B, -1, -1)
        obs_all = obs.unsqueeze(1).expand(-1, A, -1)
        goal_all = goal.unsqueeze(1).expand(-1, A, -1)

        obs_flat = obs_all.reshape(B * A, self.obs_dim)
        act_flat = act_all.reshape(B * A, A)
        goal_flat = goal_all.reshape(B * A, self.goal_dim)

        q_heads = self.forward(obs_flat, act_flat, goal_flat, return_all_heads=True)
        return q_heads.view(B, A, self.num_heads)

    @torch.no_grad()
    def q_val_for_argmax_action(self, obs, goal):
        q_heads = self.q_values_all_actions_all_heads(obs, goal)
        return q_heads.mean(dim=-1)

    @torch.no_grad()
    def q_ucb_all_actions(self, obs, goal, ucb_beta=0.1):
        q_heads = self.q_values_all_actions_all_heads(obs, goal)
        q_mean = q_heads.mean(dim=-1)
        q_std = q_heads.std(dim=-1, unbiased=False)
        q_ucb = q_mean + ucb_beta * q_std
        return q_ucb, q_mean, q_std, q_heads

In [ ]:
MAZE_LAYOUT = [
    [1,1,1,1,1,1,1,1,1,1,1],
    [1,0,0,0,0,1,0,0,0,0,1],
    [1,0,1,1,0,1,0,1,1,0,1],
    [1,0,1,0,0,0,0,0,1,0,1],
    [1,0,1,0,1,1,1,0,1,0,1],
    [1,0,0,0,1,0,0,0,1,0,1],
    [1,1,1,0,1,0,1,1,1,0,1],
    [1,0,0,0,0,0,1,0,0,0,1],
    [1,0,1,1,1,0,1,1,1,0,1],
    [1,0,0,0,1,0,0,0,0,0,1],
    [1,1,1,1,1,1,1,1,1,1,1],
]

def make_env(goal=(9, 9), slip_prob=0.00, max_horizon=100):
    base = MazeGridWorld(
        maze=MAZE_LAYOUT,
        max_episode_steps=max_horizon,
    )
    env = MazeGoalWrapper(
        base,
        goal_position=goal,
        goal_reward=1.0,
        step_reward=0.0,
        slip_prob=slip_prob,
        reward_mode="simple",
    )
    return env

env = make_env(goal=(9, 9))
obs, info = env.reset()

img = env.unwrapped.render()
goal = env.goal_position

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.scatter(
    goal[0] * 40 + 20,
    goal[1] * 40 + 20,
    c="lime",
    s=180,
    marker="*",
    edgecolors="black",
)
plt.title(f"Maze with goal at {goal}")
plt.axis("off")
plt.show()

In [ ]:
@torch.no_grad()
def select_action_ucb(q_network, obs, goal_t_single, device, ucb_beta=1.0):
    obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    q_ucb, q_mean, q_std, _ = q_network.q_ucb_all_actions(obs_t, goal_t_single, ucb_beta=ucb_beta)
    action = int(q_ucb.argmax(dim=-1).item())
    return action, q_mean.squeeze(0), q_std.squeeze(0)

In [ ]:
def dqn_train_ensemble(
    seed: int = 42,
    q_network=None,
    q_target_network=None,
    env=None,
    buffer_capacity=None,
    lr=None,
    obs_dim=None,
    device=None,
    total_steps=100000,
    warmup_steps=5000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    train_freq=4,
    goal=None,
    params=None,
    regulariser=None,
    reg_alpha=500,
    embedding_memory=None,
    reference_params=None,
    fisher_diag=None,
    sa_reg_prefix_filter="sa_encoder",
    sigreg=1,
    td_steps=1,
    make_env=None,
    early_stop_reward=0.99,
    early_stop_patience=3,
    enable_early_stop=True,
    ucb_beta=0.1,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_steps=80000,
    use_ucb=True,
):
    if q_network is None:
        raise ValueError("q_network must be provided")
    if q_target_network is None:
        raise ValueError("q_target_network must be provided")
    if env is None:
        raise ValueError("env must be provided")
    if goal is None:
        raise ValueError("goal must be provided")
    if buffer_capacity is None:
        raise ValueError("buffer_capacity must be provided")
    if lr is None:
        raise ValueError("lr must be provided")
    if make_env is None:
        raise ValueError("make_env function must be provided")

    if device is None:
        device = next(q_network.parameters()).device

    if obs_dim is None:
        obs_dim = int(env.observation_space.shape[0])

    if embedding_memory is None:
        embedding_memory = {}

    set_seed(seed)

    if params is None:
        opt = optim.Adam(q_network.parameters(), lr=lr)
    else:
        opt = optim.Adam(params, lr=lr)

    buffer = TrajectoryReplayBufferDiscrete(buffer_capacity, obs_dim, 1, device=device)

    goal_arr = np.array(goal, dtype=np.float32)
    goal_t_single = torch.tensor(goal_arr, dtype=torch.float32, device=device).unsqueeze(0)

    obs, _ = env.reset()
    global_step = 0
    eval_returns = []
    start_time = time.perf_counter()
    min_steps = None
    min_time = None
    success_streak = 0

    ortho_loss = torch.tensor(0.0, device=device)
    sigreg_loss_val = torch.tensor(0.0, device=device)
    weight_loss = torch.tensor(0.0, device=device)
    ewc_loss = torch.tensor(0.0, device=device)
    loss = torch.tensor(0.0, device=device)
    td_loss = torch.tensor(0.0, device=device)

    num_actions = env.action_space.n
    num_heads = q_network.num_heads
    eps = eps_start

    while global_step < total_steps:
        if global_step < warmup_steps:
            action = env.action_space.sample()
        else:
            frac = min(1.0, (global_step - warmup_steps) / eps_decay_steps)
            eps = eps_start + frac * (eps_end - eps_start)

            if np.random.random() < eps:
                action = env.action_space.sample()
            else:
                obs_t_single = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    if use_ucb:
                        q_act, _, _, _ = q_network.q_ucb_all_actions(
                            obs_t_single, goal_t_single, ucb_beta=ucb_beta
                        )
                    else:
                        q_act = q_network.q_val_for_argmax_action(obs_t_single, goal_t_single)
                    action = int(q_act.argmax(dim=-1).item())

        next_obs, rew, term, trunc, _ = env.step(action)
        done = term or trunc

        buffer.add_transition(obs, action, rew, next_obs, done)
        obs = next_obs
        global_step += 1

        if done:
            obs, _ = env.reset()

        if len(buffer) >= warmup_steps and global_step % train_freq == 0:
            batch = buffer.sample(batch_size)

            obs_t = batch.obs
            act_t = batch.actions.long()
            rew_t = batch.rewards
            next_obs_t = batch.next_obs
            term_t = batch.terminated
            trunc_t = batch.truncated
            done_t = torch.clamp(term_t + trunc_t, 0.0, 1.0)

            goal_batch = goal_t_single.expand(obs_t.shape[0], -1)
            B = obs_t.shape[0]

            with torch.no_grad():
                next_q_heads = q_target_network.q_values_all_actions_all_heads(next_obs_t, goal_batch)   # [B, A, K]
                next_actions_per_head = next_q_heads.argmax(dim=1)                                        # [B, K]

                batch_idx = torch.arange(B, device=device).unsqueeze(1)                                   # [B,1]
                head_idx = torch.arange(num_heads, device=device).unsqueeze(0)                            # [1,K]
                next_q_per_head = next_q_heads[batch_idx, next_actions_per_head, head_idx]                # [B,K]

                gamma_final = gamma ** td_steps
                target = rew_t + gamma_final * (1.0 - done_t) * next_q_per_head                           # [B,K]

            act_onehot = F.one_hot(act_t.squeeze(-1), num_classes=num_actions).float()
            current_q_heads = q_network(obs_t, act_onehot, goal_batch, return_all_heads=True)             # [B,K]

            td_loss = F.mse_loss(current_q_heads, target)

            if sigreg is not None:
                act_onehot_all = F.one_hot(
                    torch.arange(num_actions, device=device),
                    num_classes=num_actions
                ).float()
                act_onehot_all = act_onehot_all.unsqueeze(0).expand(B, -1, -1)

                obs_rep = obs_t.unsqueeze(1).expand(-1, num_actions, -1)
                obs_flat = obs_rep.reshape(B * num_actions, obs_dim)
                act_flat = act_onehot_all.reshape(B * num_actions, num_actions)

                phi_all = q_network.encode_state_action(obs_flat, act_flat)
                sigreg_loss_val = sigreg_loss(phi_all)
            else:
                sigreg_loss_val = torch.tensor(0.0, device=device)

            if fisher_diag is not None:
                ewc_loss = ewc_regulariser_loss(
                    q_network,
                    reference_params=reference_params,
                    fisher_diag=fisher_diag,
                    prefix_filter=sa_reg_prefix_filter,
                )
            else:
                ewc_loss = torch.tensor(0.0, device=device)

            loss = td_loss
            if regulariser is not None and regulariser == "repulsion":
                loss = loss + reg_alpha * ortho_loss + 0.1 * sigreg_loss_val + 10000 * ewc_loss

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_network.parameters(), 10.0)
            opt.step()

            for p, p_tgt in zip(q_network.parameters(), q_target_network.parameters()):
                p_tgt.data.mul_(1.0 - tau).add_(tau * p.data)

        if global_step % 1000 == 0:
            eval_env = make_env(goal)
            goal_eval_t = torch.tensor(np.array(goal, dtype=np.float32), dtype=torch.float32, device=device).unsqueeze(0)

            def eval_policy(o):
                o_t = torch.tensor(o, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    q_vals = q_network.q_val_for_argmax_action(o_t, goal_eval_t)
                    return int(q_vals.argmax(dim=-1).item())

            mean_ret, mean_len = evaluate_policy(eval_env, eval_policy, episodes=8)
            eval_returns.append((global_step, mean_ret))

            with torch.no_grad():
                probe_obs = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                _, q_mean_probe, q_std_probe, _ = q_network.q_ucb_all_actions(
                    probe_obs, goal_t_single, ucb_beta=ucb_beta
                )
                avg_std = q_std_probe.mean().item()

            print(
                f"[DQN-ensemble] step={global_step:7d}"
                f" | eps={eps:.3f}"
                f" | beta={ucb_beta:.3f}"
                f" | eval_return={mean_ret:.3f}"
                f" | eval_len={mean_len:.1f}"
                f" | td={td_loss.item():.4f}"
                f" | std={avg_std:.4f}"
                f" | sig={sigreg_loss_val.item():.4f}"
            )

            if mean_ret >= early_stop_reward:
                success_streak += 1
            else:
                success_streak = 0

            if enable_early_stop and success_streak >= early_stop_patience:
                min_steps = global_step
                min_time = time.perf_counter() - start_time
                eval_env.close()
                break

            eval_env.close()

    goal_tensor = torch.tensor(np.array(goal, dtype=np.float32), dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        psi_z = q_network.encode_goal(goal_tensor)
        task_embedding = psi_z.squeeze(0).cpu().numpy()

    obs_probe_np = np.array([5.0, 5.0], dtype=np.float32)
    base_env = env.unwrapped if hasattr(env, "unwrapped") else env
    env_action_names = getattr(base_env, "action_names", ["Up", "Down", "Left", "Right"])
    act_probe_idx = env_action_names.index("Up")

    sa_embedding_mean = extract_mean_sa_embedding(
        q_network=q_network,
        buffer=buffer,
        num_actions=num_actions,
        batch_size=256,
        device=device,
        as_numpy=True,
    )

    sa_embedding_fixed = extract_fixed_probe_sa_embedding(
        q_network=q_network,
        obs_probe=obs_probe_np,
        act_probe_idx=act_probe_idx,
        num_actions=num_actions,
        device=device,
        as_numpy=True,
    )

    sa_batch_final = extract_sa_batch_for_isotropy(
        q_network=q_network,
        buffer=buffer,
        num_actions=num_actions,
        batch_size=1024,
        device=device,
        as_numpy=True,
    )

    env.close()
    return (
        q_network,
        q_target_network,
        eval_returns,
        min_steps,
        min_time,
        task_embedding,
        sa_embedding_mean,
        sa_embedding_fixed,
        sa_batch_final,
        buffer,
    )

## Training Loop across goals

first loop = retrain from scratch for every goal

second loop = retrain from previous task weights as initialisation

In [ ]:
from copy import deepcopy
import numpy as np
import torch

SEEDS = [42]
PASS = 1
GOALS = [(9, 9), (7, 7), (3, 5), (3, 9), (5, 5), (3, 6), (9, 1), (1, 1), (7, 4), (9, 8), (9, 7)]

BUFFER_CAPACITY = 100000
LR = float(1e-3)
NUM_HEADS = 3
UCB_BETA = 0.3

sa_keywords_local = ["sa_encoder"]
goal_keywords_local = ["goal_encoder"]

overall_results = {
    goal: {
        "eval_returns": [],
        "min_steps": [],
        "min_time": [],
        "task_embeddings": [],
        "sa_embeddings": [],
        "sa_fixed_probe_embeddings": [],
        "sa_batches_final": [],
        "num_heads": NUM_HEADS,
        "ucb_beta": UCB_BETA,
    }
    for goal in GOALS
}

for seed in SEEDS:
    print(f"\n================ SEED {seed} ================\n")
    set_seed(seed)

    env_tmp = make_env(goal=GOALS[0])
    obs_dim = env_tmp.observation_space.shape[0]
    num_actions = env_tmp.action_space.n
    env_tmp.close()

    q_net_base = FactorisedDQNQNetworkEnsemble(
        obs_dim=obs_dim,
        num_actions=num_actions,
        goal_dim=2,
        hidden_dim=128,
        rep_dim=64,
        num_heads=NUM_HEADS,
    ).to(DEVICE)

    q_target_base = FactorisedDQNQNetworkEnsemble(
        obs_dim=obs_dim,
        num_actions=num_actions,
        goal_dim=2,
        hidden_dim=128,
        rep_dim=64,
        num_heads=NUM_HEADS,
    ).to(DEVICE)

    q_target_base.load_state_dict(q_net_base.state_dict())
    for p in q_target_base.parameters():
        p.requires_grad_(False)

    seed_task_embedding_memory = []
    seen_goal_labels = []
    weight_history = []

    prev_q_net = None
    prev_q_target = None

    for pass_num in range(PASS):
        print(f"\n===== PASS {pass_num + 1} / {PASS} =====\n")

        for goal_idx, goal in enumerate(GOALS):
            print(f"\n----- seed={seed}, pass={pass_num + 1}, goal={goal} -----\n")

            if pass_num == 0 or goal_idx == 0:
                q_net = deepcopy(q_net_base)
                q_target = deepcopy(q_target_base)
                q_target.load_state_dict(q_net.state_dict())
                for p in q_target.parameters():
                    p.requires_grad_(False)
            else:
                if prev_q_net is None or prev_q_target is None:
                    raise ValueError("Previous Q-networks are not available for pass > 0.")
                q_net = deepcopy(prev_q_net)
                q_target = deepcopy(prev_q_target)
                q_target.load_state_dict(q_net.state_dict())
                for p in q_target.parameters():
                    p.requires_grad_(False)

            weight_history.append(
                collect_weight_snapshot(
                    qnet=q_net,
                    goal_label=goal,
                    stage_label=f"goal_{goal_idx}_before_train_{goal}_pass_{pass_num + 1}",
                    sa_keywords_local=sa_keywords_local,
                    goal_keywords_local=goal_keywords_local,
                    max_samples_per_group=40000,
                )
            )

            (
                q_network,
                q_target_network,
                eval_returns,
                min_steps,
                min_time,
                task_embedding,
                sa_embedding_mean,
                sa_embedding_fixed,
                sa_batch_final,
                buffer,
            ) = dqn_train_ensemble(
                seed=seed,
                q_network=q_net,
                q_target_network=q_target,
                env=make_env(goal=goal),
                obs_dim=obs_dim,
                buffer_capacity=BUFFER_CAPACITY,
                lr=LR,
                goal=goal,
                device=DEVICE,
                embedding_memory=seed_task_embedding_memory,
                regulariser=None,
                reg_alpha=1,
                make_env=make_env,
                ucb_beta=UCB_BETA,
            )

            weight_history.append(
                collect_weight_snapshot(
                    qnet=q_network,
                    goal_label=goal,
                    stage_label=f"goal_{goal_idx}_after_train_{goal}_pass_{pass_num + 1}",
                    sa_keywords_local=sa_keywords_local,
                    goal_keywords_local=goal_keywords_local,
                    max_samples_per_group=40000,
                )
            )

            visualise_q_table(
                goal=goal,
                q_network=q_network,
                eval_returns=eval_returns,
                device=DEVICE,
                make_env=make_env,
            )

            visualise_embeddings(
                goal=goal,
                q_network=q_network,
                device=DEVICE,
                make_env=make_env,
            )

            seed_task_embedding_memory.append(task_embedding)
            seen_goal_labels.append(str(goal))
            print_goal_embedding_similarity(
                seed_task_embedding_memory,
                goal_labels=seen_goal_labels,
            )

            prev_q_net = deepcopy(q_network)
            prev_q_target = deepcopy(q_target_network)

            overall_results[goal]["eval_returns"].append(eval_returns)
            overall_results[goal]["min_steps"].append(min_steps)
            overall_results[goal]["min_time"].append(min_time)
            overall_results[goal]["task_embeddings"].append(task_embedding)
            overall_results[goal]["sa_embeddings"].append(sa_embedding_mean)
            overall_results[goal]["sa_fixed_probe_embeddings"].append(sa_embedding_fixed)
            overall_results[goal]["sa_batches_final"].append(sa_batch_final)

In [ ]:
goals = list(overall_results.keys())

min_steps_per_goal = []
min_time_per_goal = []
used_goals = []

for goal in goals:
    steps_raw = overall_results[goal]["min_steps"]
    time_raw  = overall_results[goal]["min_time"]

    steps_arr = np.array(steps_raw).flatten()
    time_arr  = np.array(time_raw).flatten()

    # Skip goals with no data
    if steps_arr.size == 0 or time_arr.size == 0:
        print(f"[WARN] Skipping goal {goal}: empty min_steps or min_time")
        continue

    min_steps_per_goal.append(steps_arr.min())
    min_time_per_goal.append(time_arr.min())
    used_goals.append(goal)

if not min_steps_per_goal:
    raise ValueError("No goals with non-empty min_steps/min_time found.")

x = np.arange(len(used_goals))

fig, ax1 = plt.subplots(figsize=(8,4))

ax1.set_title("Minimum steps and time to reach a good return (per goal)")

# Left y-axis: min steps
line1 = ax1.plot(
    x, min_steps_per_goal,
    label="Min Steps",
    color="tab:blue",
    marker="o",
)
ax1.set_xlabel("Goal Index")
ax1.set_ylabel("Min Steps", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

# Right y-axis: min time
ax2 = ax1.twinx()
line2 = ax2.plot(
    x, min_time_per_goal,
    label="Min Time",
    color="tab:orange",
    marker="o",
)
ax2.set_ylabel("Min Time", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

# Label x-axis with actual goal IDs
ax1.set_xticks(x)
ax1.set_xticklabels([str(g) for g in used_goals], rotation=45)

# Combined legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="best")

fig.tight_layout()
plt.show()

## HTML visualisation

In [ ]:
for goal, data in overall_results.items():
    print(goal)
    print("task_embeddings:", len(data.get("task_embeddings", [])))
    print("sa_fixed_probe_embeddings:", len(data.get("sa_fixed_probe_embeddings", [])))
    print("sa_batches_final:", len(data.get("sa_batches_final", [])))

dashboard = plot_full_embedding_dashboard_html(
    overall_results=overall_results,
    qnet=q_net,
    mode="all",
    save_html="plots/full_embedding_dashboard_maze_baseline.html",
    weights_his=weight_history
)

print(dashboard["save_html"])
print(dashboard["isotropy_metrics"])